In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q01/annotated/checkpoints/post_cell_0.pickle

In [ ]:
Out.clear()

In [ ]:
%%RecordEvent
%%time
### cell 1 ###


date = pd.Timestamp("1998-09-02")
lineitem_filtered = lineitem.loc[
    :,
    [
        "L_QUANTITY",
        "L_EXTENDEDPRICE",
        "L_DISCOUNT",
        "L_TAX",
        "L_RETURNFLAG",
        "L_LINESTATUS",
        "L_SHIPDATE",
        "L_ORDERKEY",
    ],
]
sel = lineitem_filtered.L_SHIPDATE <= date
lineitem_filtered = lineitem_filtered[sel]
lineitem_filtered["AVG_QTY"] = lineitem_filtered.L_QUANTITY
lineitem_filtered["AVG_PRICE"] = lineitem_filtered.L_EXTENDEDPRICE
lineitem_filtered["DISC_PRICE"] = lineitem_filtered.L_EXTENDEDPRICE * (
    1 - lineitem_filtered.L_DISCOUNT
)
lineitem_filtered["CHARGE"] = (
    lineitem_filtered.L_EXTENDEDPRICE
    * (1 - lineitem_filtered.L_DISCOUNT)
    * (1 + lineitem_filtered.L_TAX)
)
gb = lineitem_filtered.groupby(["L_RETURNFLAG", "L_LINESTATUS"], as_index=False)[
    [
        "L_QUANTITY",
        "L_EXTENDEDPRICE",
        "DISC_PRICE",
        "CHARGE",
        "AVG_QTY",
        "AVG_PRICE",
        "L_DISCOUNT",
        "L_ORDERKEY",
    ]
]
total = gb.agg(
    {
        "L_QUANTITY": "sum",
        "L_EXTENDEDPRICE": "sum",
        "DISC_PRICE": "sum",
        "CHARGE": "sum",
        "AVG_QTY": "mean",
        "AVG_PRICE": "mean",
        "L_DISCOUNT": "mean",
        "L_ORDERKEY": "count",
    }
)

In [ ]:
%%RecordEvent
orig_output = Out.get(5)

In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q01/annotated/checkpoints/post_cell_1.pickle